# Modular experiment pipeline


In [ ]:
%pip install -q torch transformers bm25s scikit-learn pandas numpy matplotlib sentence-transformers tqdm

In [ ]:
from config import *
from data import load_data,validate_data,split_train_validation
from evaluation import evaluate_recall
from bm25_retrieval import BM25Retriever
from dense_retrieval import DenseRetriever
from training import build_training_examples,fine_tune_model,save_model
from submission import write_submission
set_seed()
catalog,train_gold,test_queries=load_data()
catalog_ids,catalog_id_set,test_ids=validate_data(catalog,train_gold,test_queries)
train_split,val_split=split_train_validation(train_gold)
qrels_train={q["query_id"]:[q["item_id"]] for q in train_gold}
qrels_val={q["query_id"]:[q["item_id"]] for q in val_split}

## BM25

In [ ]:
bm25=BM25Retriever(catalog,catalog_ids)
bm25_train_preds=bm25.retrieve(train_gold,TOP_K)
print("BM25",evaluate_recall(bm25_train_preds,qrels_train,TOP_K))
bm25_test_preds=bm25.retrieve(test_queries,TOP_K)

## Pretrained dense retrieval

In [ ]:
dense=DenseRetriever.from_pretrained(catalog,catalog_ids,max_seq_length=BASE_MAX_SEQ_LENGTH)
dense_train_preds=dense.retrieve(train_gold,TOP_K)
print("Pretrained dense",evaluate_recall(dense_train_preds,qrels_train,TOP_K))
dense_test_preds=dense.retrieve(test_queries,TOP_K)

## Fine-tuning

In [ ]:
bm25_train_candidates=bm25.retrieve(train_split,TOP_K)
examples=build_training_examples(train_split,bm25_train_candidates,catalog)
ft_model=fine_tune_model(examples)
save_model(ft_model)
ft_retriever=DenseRetriever(ft_model,catalog,catalog_ids)
ft_val_preds=ft_retriever.retrieve(val_split,TOP_K)
print("Fine-tuned validation",evaluate_recall(ft_val_preds,qrels_val,TOP_K))
ft_test_preds=ft_retriever.retrieve(test_queries,TOP_K)

## Submission

In [ ]:
submission_path=write_submission(test_queries,ft_test_preds,catalog_id_set)
print(f"Saved: {submission_path}")